In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import anndata as ad

In [ ]:
sc._settings.ScanpyConfig.n_jobs = 8

In [ ]:
def z_score_normalization(data):
    return (data - np.mean(data, axis=0)) / np.std(data, axis=0)

def reorder_columns(df, col_changed, col_position) -> pd.DataFrame:
    """
    Reordering colums. The second input can either be an integer for index or it can be a reference column name. If reference column name is chosen, the column will be placed after the reference column.
    """
    if not isinstance(df, pd.DataFrame):
        raise ValueError("The first argument must be a pandas DataFrame")
    if not isinstance(col_changed, str):
        raise ValueError("The second argument must be a string representing a column name")
    if not isinstance(col_position, (str, int)):
        raise ValueError("The third argument must be either a string representing a column name or an integer representing a column index")
    if isinstance(col_position, int):
        cols = df.columns.tolist()
        if col_changed in cols and col_position <= len(cols) + 1:
            cols.remove(col_changed)
        
            index = col_position
            cols.insert(index, col_changed)
            
            df = df[cols]
    else:
        cols = df.columns.tolist()
        if col_changed in cols and col_position in cols:
            cols.remove(col_changed)
            index = cols.index(col_position)
            cols.insert(index+1, col_changed)
            
            df = df[cols]
    return df

In [ ]:
df = pd.read_csv('/Users/lukashat/Documents/PhD_Schapiro/Projects/phenotype_benchmark/datasets/cycif_ovca/quantification/cycif_single_cell_spatial_25.csv', index_col=0)

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
len(df.Patient_code_final.unique())

In [ ]:
len(df.Sample_code_final.unique())

In [ ]:
df.Sample_code_final.unique()

In [ ]:
df.GlobalCellType2.value_counts()

In [ ]:
df.drop(columns=['neighbordood_cluster2', 'Stage', ], inplace=True) 
# x/y position are swapped in the original data
df.rename(columns={'GlobalCellType2': 'cell_type', 'X.Position': 'y', 'Y.Position': 'x', 'ID': 'cell_id', 'Sample_code_final': 'sample_id'}, inplace=True)

In [ ]:
df = reorder_columns(df, 'cell_type', 'sample_id')

In [ ]:
df = reorder_columns(df, 'cell_id', 'y')

In [ ]:
df

In [ ]:
marker = 'HE4'
sns.histplot(data=df, x=marker, bins=500, hue='sample_id', element='step', stat='density', common_norm=False)
plt.xlabel(marker)
plt.ylabel('Density')
#plt.xlim(0, 0.02)
plt.title(f'{marker} distribution by sample_id')
plt.show()

In [ ]:
cd3d_median = df.groupby('sample_id')['CD3d'].median().median()
df.groupby('sample_id')['CD3d'].median().plot(kind='bar', ylim=[0, 0.02])
plt.axhline(cd3d_median, color='red', linestyle='--', label=f'Median: {cd3d_median:.4f}')
plt.legend()
plt.title('CD3d median by sample_id')

In [ ]:
desmin_median = df.groupby('sample_id')['Desmin'].median().median()
df.groupby('sample_id')['Desmin'].median().plot(kind='bar', ylim=[0, 0.02])
plt.axhline(desmin_median, color='red', linestyle='--', label=f'Median: {desmin_median:.4f}')
plt.legend()

In [ ]:
remove_samples = ['S002_primary', 'S003_primary', 'S004_primary', 'S007_primary', 'S009_interval', 'S009_primary', 'S013_primary', 'S006_primary', 'S005_primary', 'S008_interval', 'S15_interval', 'S14_interval', 'S12_interval', 'S11_interval', 'S10_interval', 'S01_primary', 'S02_primary', 'S03_primary', 'S04_primary', 'S05_primary', 'S06_primary', 'S07_primary', 'S08_interval', 'S09_interval', 'S10_interval', 'S11_interval', 'S12_interval', 'S13_primary']
df = df[~df['sample_id'].isin(remove_samples)]

In [ ]:
desmin_median = df.groupby('sample_id')['Desmin'].median().median()
df.groupby('sample_id')['Desmin'].median().plot(kind='bar', ylim=[0, 0.02])
plt.axhline(desmin_median, color='red', linestyle='--', label=f'Median: {desmin_median:.4f}')
plt.legend()

In [ ]:
X_columns = df.columns[:df.columns.get_loc('x')]
obs_columns =df.columns[df.columns.get_loc('x'):]
adata = ad.AnnData(
    X=df[X_columns],
    obs=df[obs_columns],
    var=pd.DataFrame(index=X_columns)
)

In [ ]:

adata.raw = adata.copy()
#adata.X = np.log1p(adata.X)
adata.layers['zscore'] = z_score_normalization(adata.X)
adata.obs['cell_type'] = adata.obs['cell_type'].astype('category')


In [ ]:
sc.tl.dendrogram(adata, groupby='cell_type')
sc.pl.matrixplot(adata, var_names=adata.var_names, groupby='cell_type', cmap='vlag', dendrogram=True, use_raw=False, standard_scale='var'
                 )

Cancer and emt do not really differ in thei expression profile. Also visually unclear what distinguishes them

# Harmonize celltype labels

In [ ]:
df.columns

In [ ]:
df['cell_type'].value_counts()

In [ ]:
df['cell_type'] = df['cell_type'].replace({'Epithelial': 'Cancer', 'Desmin.positive.cell': 'undefined', 'IBA1.CD163.Macrophages': 'M2_Macrophage', 'EMT': 'Cancer_EMT', 
                                           'SMA.CD31.positive.cell': 'EndoMT', 'Proliferating.epithelial': 'Cancer', 'CD8.T.cells': 'CD8+_T_cell', 'Myofibroblast': 'Myofibroblasts',
                                            'CD11c.myeloid': 'undefined', 'IBA1.CD11c.Macrophages': 'M1_Macrophage', 'Proliferating.EMT': 'Cancer_EMT', 
                                            'SMA.Desmin.positive.cell': 'undefined', 'CD4.T.cells': 'CD4+_T_cell', 'Endothelial.cell': 'Endothelial', 
                                            'FOXP3.CD4.Tregs': 'Treg', 'CD163.Macrophages': 'M2_Macrophage','Bcell': 'B_cell', 'Cancer_EMT': 'Cancer', 'EndoMT': 'vSMC'})

df['cell_type'].value_counts()

In [ ]:
df['cell_type'].nunique()

# Implement different levels of granularity

In [ ]:
df['level_2_cell_type'] = df['cell_type']
df['level_2_cell_type'] = df['level_2_cell_type'].replace({'CD8+_T_cell':'Lymphoid_immune', 'M2_Macrophage':'Myeloid_immune', 'CD4+_T_cell':'Lymphoid_immune','Treg':'Lymphoid_immune', 'Dendritic_cell':'Myeloid_immune', 'B_cell':'Lymphoid_immune',
                                                           'Cancer_EMT':'Cancer', 'EndoMT':'Vascular', 'Myofibroblasts':'Fibroblast', 'M1_Macrophage':'Myeloid_immune', 'Dendritic_cell':'Myeloid_immune',
                                                           'Endothelial':'Vascular'})
df['level_2_cell_type'].value_counts()

In [ ]:
df['level_1_cell_type'] = df['level_2_cell_type']
df['level_1_cell_type'] = df['level_1_cell_type'].replace({'Lymphoid_immune':'Immune', 'Myeloid_immune':'Immune', 'Fibroblast':'Stromal', 'Vascular':'Stromal'})
df['level_1_cell_type'].value_counts()

In [ ]:
df = reorder_columns(df, 'cell_type', 'level_1_cell_type')
df = reorder_columns(df, 'level_2_cell_type', 'level_1_cell_type')
df 

In [ ]:
df['sample_id'] = df['sample_id'].str.replace('primary', 'chemonaive')
df['sample_id'] = df['sample_id'].str.replace('interval', 'IDS')
df

In [ ]:
df['sample_id'].unique()

In [ ]:
df['sample_id'].nunique()

In [ ]:
df

In [ ]:
df.to_csv('/Users/lukashat/Documents/PhD_Schapiro/Projects/phenotype_benchmark/datasets/cycif_ovca/quantification/processed/cycif_ovca_quantification.csv')

# Channel names after correspondence with lead author

In [ ]:
channel_names=['DNA1', 'Rabbit', 'Goat', 'Mouse', 'DNA2', 'TAZ', 'CD207', 'SNAT1', 'DNA3', 'CD163', 'CD57', 'CD20', 'DNA4', 'Annexin', 'pSTAT1', 'KRAS', 'DNA5', 'CD4', 'pERK', 'CD8a', 'DNA6', 'CD45RO', 'FOXP3', 'CD3d', 'DNA7', 'TIM3', 'oldCD68', 'Desmin', 'DNA8', 'CD15','oldCD11b', 'FOXOA3', 'DNA9', 'HE4', 'CD11c', 'yH2AX', 'DNA10', 'Ki67', 'Vimentin', 'MHCII', 'DNA11', 'LaminB1', 'CK7', 'MHCI', 'DNA12', 'Ecadherin', 'SMA', 'CD31', 'DNA13', 'IBA1', 'CD68', 'CD11b']

In [ ]:
with open('/Users/lukashat/Documents/PhD_Schapiro/Projects/phenotype_benchmark/datasets/cycif_ovca/markers.txt', 'w') as f:
    for channel in channel_names:
        f.write(f"{channel}\n")

In [ ]:
df = pd.read_csv('/Users/lukashat/Documents/PhD_Schapiro/Projects/phenotype_benchmark/datasets/cycif_ovca/quantification/cycif_ovca_quantification.csv', index_col=0)

In [ ]:
X_columns = df.columns[:df.columns.get_loc('y')]
obs_columns =df.columns[df.columns.get_loc('y'):]
adata = ad.AnnData(
    X=df[X_columns],
    obs=df[obs_columns],
    var=pd.DataFrame(index=X_columns)
)

In [ ]:
adata.obs['cell_type'] = adata.obs['cell_type'].astype('category')

In [ ]:
sc.tl.dendrogram(adata, groupby='cell_type')
sc.pl.matrixplot(adata, var_names=adata.var_names, groupby='cell_type', cmap='vlag', dendrogram=True, use_raw=False, standard_scale='var'
                 )

In [ ]:
df.columns

In [ ]:
adata.X.max()

In [ ]:
df

In [ ]:
df_sample = df.sample(frac=0.4, random_state=42)
df_sample.to_csv('/Users/lukashat/Documents/PhD_Schapiro/Projects/phenotype_benchmark/datasets/cycif_ovca/quantification/cycif_ovca_quantification_sample40.csv')

In [ ]:
df = pd.read_csv('/Users/lukashat/Documents/PhD_Schapiro/Projects/phenotype_benchmark/datasets/cycif_ovca/quantification/processed/cycif_ovca_quantification.csv', index_col=0)

In [ ]:
df['sample_id'].unique()